In [1]:
# 8_cluster_national_level.ipynb
#
# Produces LA-level personas using cluster definitions fit at the NATIONAL level.
#
# Key difference from 7_cluster_local_level:
#   • K-Means is fit ONCE on ALL UKHLS respondents (o_indresp_derived.pkl, ~19K rows)
#     — cluster definitions are shared across every LA.
#   • The same tribe labels (e.g. 'Employed 1', 'Retired 2') mean the same thing
#     in every area, enabling cross-LA comparison.
#   • USE_FOUR_LA_SUBSET / UNIT_FILTER controls OUTPUT scope only — clustering itself
#     always uses all national UKHLS data.
#
# Pipeline:
#   Phase 1 — Fit national clusters (UKHLS derived pickle, ~19K rows):
#     For each employment group, normalise features and run K-Means.
#     Store: pidp → (group, tribe_label).
#
#   Phase 2 — Profile per target LA (reads synthetic parquet filtered per LA):
#     Join each LA's rows with national assignments, compute DNA profiles.
#     Append to output CSV.
#
# Output: data/8_cluster_national_level/LA_london_national_clusters.csv
#   Same columns as 7_cluster_local_level — fully compatible with the API (mode=national).

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import USE_TEST_DATA, DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
_cv.reload_config_variables()
importlib.reload(_cc)

import pandas as pd
import numpy  as np
from pathlib import Path
from tqdm    import tqdm

import data_pipeline.helpers.normalise as normalise
import data_pipeline.helpers.cluster as cf
importlib.reload(normalise)
importlib.reload(cf)

from data_pipeline.config_variables import (
    CLUSTER_VARS, SUMMARY_VARS, VARIABLE_MAP,
    CATEGORICAL_VARS, CATEGORY_MAPS, CONTINUOUS_VARS,
)
from data_pipeline.config_cluster import WAVE, GROUPS, MAX_TOTAL_CLUSTERS

# ── Config ────────────────────────────────────────────────────────────────────
# FEATURE_PKL  : UKHLS derived features — ALL respondents used for fitting clusters.
# SYNPOP_PARQUET: synthetic population — filtered per LA in Phase 2.
FEATURE_PKL    = f"../{DATA_FOLDER}/5_derive_variables/o_indresp_derived.pkl"
SYNPOP_PARQUET = f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet"
OUTPUT_DIR     = Path(f"../{DATA_FOLDER}/8_cluster_national_level")
GEO_CSV        = Path(f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv")
MIN_CLUSTER_SIZE = 5

# Output scope: which LAs to produce profiles for.
# Clustering itself is always national (all UKHLS respondents).
UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else "london"

for p in [FEATURE_PKL, SYNPOP_PARQUET]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} — run pipeline steps first.")

_filter_tag = "_london" if UNIT_FILTER == "london" or USE_FOUR_LA_SUBSET else "_custom"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / f"LA{_filter_tag}_national_clusters.csv"
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()
    print(f"Removed existing {OUTPUT_CSV.name}")

# ── LA name lookup ───────────────────────────────────────────────────────────
_la_name_map = {}
if GEO_CSV.exists():
    _geo = pd.read_csv(GEO_CSV, encoding='latin-1', usecols=['ladcd', 'ladnm']).drop_duplicates('ladcd')
    _la_name_map = _geo.set_index('ladcd')['ladnm'].to_dict()
    print(f"Loaded {len(_la_name_map)} LA names from {GEO_CSV.name}")

# ── Phase 1: Fit national clusters on ALL UKHLS respondents ─────────────────
print(f"\n── Phase 1: national cluster fitting (MAX_TOTAL_CLUSTERS={MAX_TOTAL_CLUSTERS}) ──")

df_ukhls = pd.read_pickle(FEATURE_PKL)
df_ukhls['pidp'] = df_ukhls['pidp'].astype('int64')
print(f"  Loaded {len(df_ukhls):,} UKHLS respondents × {len(df_ukhls.columns)} cols")

import pyarrow.parquet as pq
parquet_cols  = set(pq.read_schema(SYNPOP_PARQUET).names)
feature_cols  = [f"{WAVE}_{b}" for b in CLUSTER_VARS if f"{WAVE}_{b}" in df_ukhls.columns]
missing_feat  = [f"{WAVE}_{b}" for b in CLUSTER_VARS if f"{WAVE}_{b}" not in df_ukhls.columns]
print(f"  Feature columns: {len(feature_cols)} present, {len(missing_feat)} missing")
if missing_feat:
    print(f"    Missing: {missing_feat}")

# Employment group → OHE column
_JBSTAT_LOOKUP = {
    "Employed":   f"{WAVE}_jbstat_1",
    "Unemployed": f"{WAVE}_jbstat_3",
    "Retired":    f"{WAVE}_jbstat_4",
    "On leave":   f"{WAVE}_jbstat_5",
    "Student":    f"{WAVE}_jbstat_7",
    "Inactive":   f"{WAVE}_jbstat_8",
}
GROUP_COL = {
    gname: (
        _JBSTAT_LOOKUP[gname]
        if _JBSTAT_LOOKUP.get(gname) in df_ukhls.columns
        else None
    )
    for gname in GROUPS
}

# Proportional k per group
_gsizes = {}
_seen   = pd.Series(False, index=df_ukhls.index)
for _gname in GROUPS:
    _c = GROUP_COL.get(_gname)
    if _c is not None and _c in df_ukhls.columns:
        _m = df_ukhls[_c] == 1.0
    elif _c is None:
        _m = ~_seen
    else:
        continue
    _gsizes[_gname] = int(_m.sum())
    _seen |= _m
_total   = sum(_gsizes.values()) or 1
_group_k = {g: max(1, round(MAX_TOTAL_CLUSTERS * n / _total)) for g, n in _gsizes.items()}

print(f"\n  Group cluster budget:")
for g, k in _group_k.items():
    print(f"    {g:20s}: k={k}  (n={_gsizes.get(g, 0):,})")

# Fit K-Means per group — store pidp → (group, tribe_label)
pidp_assignment: dict = {}   # {pidp(int): (group_str, tribe_label_str)}
national_coeffs: dict = {}   # {group: Coefficients}  (for inspection)

assigned = pd.Series(False, index=df_ukhls.index)

for gname in GROUPS:
    col = GROUP_COL.get(gname)
    if col is not None and col in df_ukhls.columns:
        mask = df_ukhls[col] == 1.0
    elif col is None:
        mask = ~assigned   # catch-all
    else:
        continue

    group_df = df_ukhls[mask].copy()
    if group_df.empty:
        continue
    assigned |= mask

    avail_feat = [
        c for c in feature_cols
        if c in group_df.columns and group_df[c].notna().any()
    ]
    if not avail_feat:
        continue

    coeffs = normalise.fit(group_df, avail_feat)
    national_coeffs[gname] = coeffs
    group_norm = normalise.apply(group_df, coeffs)

    n       = len(group_norm)
    max_k   = min(_group_k.get(gname, 1), max(1, n // MIN_CLUSTER_SIZE))
    chosen_k = cf.best_k_by_silhouette(group_norm[avail_feat].values, max_k)
    labels  = cf.fit_kmeans(group_norm[avail_feat].values, chosen_k)

    for pidp_val, lbl in zip(group_df['pidp'].values, labels):
        tribe_label = f"{gname} {lbl + 1}" if chosen_k > 1 else gname
        pidp_assignment[int(pidp_val)] = (gname, tribe_label)

    print(f"  {gname:20s}: {n:,} respondents → {chosen_k} cluster(s)")

# Build lookup DataFrame (small — one row per UKHLS respondent)
df_assign = pd.DataFrame(
    [{'pidp': p, 'group': g, 'tribe_label': t} for p, (g, t) in pidp_assignment.items()]
)
df_assign['pidp'] = df_assign['pidp'].astype('int64')
print(f"\n  Assigned {len(df_assign):,} UKHLS respondents to",
      f"{df_assign['tribe_label'].nunique()} national tribes")

# ── Phase 2: Profile per target LA ──────────────────────────────────────────
LEVEL_COL = "ladcd"
all_unit_ids = (
    pd.read_parquet(SYNPOP_PARQUET, columns=[LEVEL_COL])[LEVEL_COL]
    .dropna().unique().tolist()
)
all_unit_ids.sort()

if UNIT_FILTER is None:
    unit_ids = all_unit_ids
elif UNIT_FILTER == "london":
    unit_ids = [u for u in all_unit_ids if str(u).startswith("E09")]
elif isinstance(UNIT_FILTER, list):
    unit_ids = [u for u in all_unit_ids if u in set(UNIT_FILTER)]
else:
    raise ValueError(f"UNIT_FILTER must be None, 'london', or a list of codes")

_filter_desc = (
    None if UNIT_FILTER is None else
    "london (33 E09 boroughs)" if UNIT_FILTER == "london" else
    f"four-LA subset ({', '.join(UNIT_FILTER)})" if USE_FOUR_LA_SUBSET else
    f"{len(UNIT_FILTER)} explicit codes"
)
print(f"\n── Phase 2: profiling {len(unit_ids)} LAs"
      + (f" [{_filter_desc}]" if _filter_desc else "") + " ──")
print(f"Output CSV: {OUTPUT_CSV}")

header_written = False

for unit_id in tqdm(unit_ids, desc="LA"):
    df_la = pd.read_parquet(
        SYNPOP_PARQUET,
        filters=[(LEVEL_COL, '==', unit_id)]
    )
    if df_la.empty:
        continue

    df_la['pidp'] = pd.to_numeric(df_la['pidp'], errors='coerce').astype('Int64')
    df_assign_m   = df_assign.copy()
    df_assign_m['pidp'] = df_assign_m['pidp'].astype('Int64')

    # Inner join: keep only synthetic rows whose pidp has a national assignment
    df_la = df_la.merge(df_assign_m, on='pidp', how='inner')
    if df_la.empty:
        continue

    unit_rows = []
    for (gname, tribe_label), sub_df in df_la.groupby(['group', 'tribe_label'], sort=False):
        row = cf.build_dna_row(
            tribe_label, sub_df, WAVE,
            SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
            continuous_vars=CONTINUOUS_VARS,
        )
        row['unit_id']       = unit_id
        row['la_name']       = _la_name_map.get(unit_id, unit_id)
        row['cluster_level'] = 'national'
        row['group']         = gname
        unit_rows.append(row)

    if unit_rows:
        unit_df = pd.DataFrame(unit_rows)
        unit_df.to_csv(
            OUTPUT_CSV,
            mode='a',
            header=not header_written,
            index=False,
        )
        header_written = True

    del df_la, unit_rows
    gc.collect()

print(f"\nDone. Results written to {OUTPUT_CSV}")
if OUTPUT_CSV.exists():
    result = pd.read_csv(OUTPUT_CSV)
    print(f"  {len(result)} tribe rows across {result['unit_id'].nunique()} units")
    print(result[['unit_id', 'cluster_level', 'group', 'tribe_label', 'size']].head(12).to_string(index=False))

    api_clusters_dir = Path("../api/data/clusters")
    api_clusters_dir.mkdir(parents=True, exist_ok=True)
    import shutil
    api_copy = api_clusters_dir / OUTPUT_CSV.name
    shutil.copy2(OUTPUT_CSV, api_copy)
    print(f"  Copied to {api_copy}")



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  


🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loaded 364 LA names from admin_geography_mappings.csv

── Phase 1: national cluster fitting (MAX_TOTAL_CLUSTERS=30) ──
  Loaded 19,618 UKHLS respondents × 65 cols
  Feature columns: 5 present, 0 missing

  Group cluster budget:
    Employed            : k=17  (n=10,847)
    Retired             : k=10  (n=6,450)
    Unemployed          : k=1  (n=630)
    Student             : k=1  (n=35

LA: 100%|██████████| 4/4 [00:00<00:00,  5.43it/s]


Done. Results written to ../data/8_cluster_national_level/LA_london_national_clusters.csv
  48 tribe rows across 4 units
  unit_id cluster_level      group tribe_label  size
E09000018      national   Employed  Employed 1 20356
E09000018      national   Employed  Employed 4 16121
E09000018      national    Retired   Retired 1 19753
E09000018      national   Employed  Employed 6  7126
E09000018      national   Employed  Employed 5 18476
E09000018      national   Employed  Employed 2 11865
E09000018      national    Retired   Retired 2  3150
E09000018      national   On leave    On leave  4720
E09000018      national   Inactive    Inactive  5999
E09000018      national   Employed  Employed 3 10248
E09000018      national Unemployed  Unemployed  8038
E09000018      national    Student     Student  3142
  Copied to ../api/data/clusters/LA_london_national_clusters.csv
